# MFCC / LPC / PLP on RAVDESS: run everything

Press **Runtime -> Run all**. At the end a `results_and_figures.zip` downloads automatically. Send that zip back. The experiment cell is the slow one (leave the tab open).

In [ ]:
!pip -q install soundfile joblib scikit-learn pandas matplotlib scipy

In [ ]:
%%writefile features.py
"""
Frame-synchronous speech features written from first principles (NumPy/SciPy only).

All three feature streams (MFCC, LPC cepstra, PLP cepstra) use the SAME framing grid:
16 kHz, 25 ms Hamming frames (400 samples), 10 ms hop (160 samples), 512-point FFT
(>= frame length, so no frame samples are discarded).
"""
import math
import numpy as np
import soundfile as sf
from scipy.fft import dct, rfft, irfft
from scipy.signal import resample_poly

SR = 16000
FRAME = 400          # 25 ms
HOP = 160            # 10 ms
NFFT = 512
N_MEL = 26
N_CEP = 13           # C1..C13 (C0 excluded)
LPC_ORDER = 18       # fs[kHz] + 2 rule of thumb
PLP_ORDER = 12
PREEMPH = 0.97
VAD_DB = 35.0        # frames within 35 dB of the loudest frame are "speech"
STREAMS = ("MFCC", "LPCC", "PLP")


# ----------------------------------------------------------------- I/O
def load_audio(path, sr=SR, rms=0.1):
    """Read a WAV file, average channels (no arbitrary channel choice), resample, RMS-normalise."""
    x, fs = sf.read(path, always_2d=True)
    x = x.mean(axis=1)
    if fs != sr:
        g = math.gcd(int(fs), int(sr))
        x = resample_poly(x, sr // g, fs // g)
    return x / (np.sqrt(np.mean(x ** 2)) + 1e-12) * rms


def add_noise(x, snr_db, rng):
    """White Gaussian noise at a given whole-utterance SNR (dB)."""
    p_sig = np.mean(x ** 2)
    p_noise = p_sig / (10 ** (snr_db / 10))
    return x + rng.standard_normal(len(x)) * math.sqrt(p_noise)


# ----------------------------------------------------------------- front end
def preemphasis(x, a=PREEMPH):
    """y[n] = x[n] - a x[n-1]; first sample kept."""
    return np.append(x[0], x[1:] - a * x[:-1])


def frame_signal(x, frame=FRAME, hop=HOP):
    """Split into frames; the final partial frame is zero-padded (so no samples are dropped)."""
    n = 1 + int(math.ceil(max(len(x) - frame, 0) / hop))
    pad = (n - 1) * hop + frame - len(x)
    xp = np.pad(x, (0, pad))
    idx = np.arange(frame)[None, :] + hop * np.arange(n)[:, None]
    return xp[idx]


def power_spectrum(frames, nfft=NFFT):
    """One-sided POWER spectrum |X|^2 of Hamming-windowed frames."""
    w = np.hamming(frames.shape[1])
    return np.abs(rfft(frames * w, n=nfft, axis=1)) ** 2


def speech_mask(frames, thresh_db=VAD_DB):
    """Simple energy VAD: frame energy within thresh_db of the loudest frame."""
    e = 10 * np.log10(np.sum(frames ** 2, axis=1) + 1e-12)
    return e > (e.max() - thresh_db)


# ----------------------------------------------------------------- MFCC
def hz2mel(f):
    return 2595.0 * np.log10(1.0 + f / 700.0)


def mel2hz(m):
    return 700.0 * (10 ** (m / 2595.0) - 1.0)


def mel_filterbank(n_filt=N_MEL, nfft=NFFT, sr=SR, fmin=0.0, fmax=None):
    """Triangular filters equally spaced on the mel scale between fmin and fmax (default 0..sr/2)."""
    fmax = fmax or sr / 2
    mpts = np.linspace(hz2mel(fmin), hz2mel(fmax), n_filt + 2)
    fpts = mel2hz(mpts)
    bins = np.floor((nfft + 1) * fpts / sr).astype(int)
    fb = np.zeros((n_filt, nfft // 2 + 1))
    for i in range(1, n_filt + 1):
        l, c, r = bins[i - 1], bins[i], bins[i + 1]
        for k in range(l, c):
            fb[i - 1, k] = (k - l) / max(c - l, 1)
        for k in range(c, r):
            fb[i - 1, k] = (r - k) / max(r - c, 1)
    return fb


_FB = mel_filterbank()


def mfcc_from_power(P, n_cep=N_CEP):
    """Power spectrum -> mel filterbank energies -> natural log (floored) -> DCT-II (orthonormal)
    -> C1..C13. No liftering, no deltas, C0 excluded."""
    E = P @ _FB.T
    logE = np.log(np.maximum(E, 1e-10))
    return dct(logE, type=2, norm="ortho", axis=1)[:, 1:n_cep + 1]


# ----------------------------------------------------------------- LPC
def autocorr_from_power(P, order, nfft=NFFT):
    """Autocorrelation r[0..order] from the power spectrum (Wiener-Khinchin)."""
    r = irfft(P, n=nfft, axis=1)[:, :order + 1]
    return r


def levinson_batch(r, order):
    """Levinson-Durbin for many frames at once. r: (T, order+1).
    Returns a (T, order+1) with a[:,0]==1 (A(z)=1+sum a_k z^-k) and the error energy e (T,)."""
    r = r.copy()
    r[:, 0] = r[:, 0] * (1 + 1e-9) + 1e-10        # tiny white-noise correction (silent frames)
    T = r.shape[0]
    a = np.zeros((T, order + 1))
    a[:, 0] = 1.0
    e = r[:, 0].copy()
    for i in range(1, order + 1):
        acc = np.sum(a[:, :i] * r[:, i:0:-1], axis=1)
        k = -acc / e
        prev = a[:, :i + 1].copy()
        a[:, 1:i + 1] = prev[:, 1:i + 1] + k[:, None] * prev[:, i - 1::-1]
        e = e * (1 - k ** 2)
        e = np.maximum(e, 1e-20)
    return a, e


def lpc_to_cep(a, n_cep=N_CEP):
    """Recursive LPC -> cepstrum (gain term excluded): c_n = -a_n - sum_{k<n} (k/n) c_k a_{n-k}."""
    T, p1 = a.shape
    p = p1 - 1
    c = np.zeros((T, n_cep + 1))
    for n in range(1, n_cep + 1):
        acc = np.zeros(T)
        for k in range(1, n):
            if n - k <= p:
                acc += (k / n) * c[:, k] * a[:, n - k]
        an = a[:, n] if n <= p else 0.0
        c[:, n] = -an - acc
    return c[:, 1:]


def lpc_from_power(P, order=LPC_ORDER):
    r = autocorr_from_power(P, order)
    return levinson_batch(r, order)


def lpc_envelope_db(a, e, nfft=NFFT, sr=SR):
    """All-pole spectral envelope in dB, 10*log10(G^2/|A(f)|^2), and its frequency axis."""
    A = rfft(a, n=nfft, axis=1)
    env = 10 * np.log10(e[:, None] / (np.abs(A) ** 2 + 1e-20) + 1e-20)
    return env, np.arange(nfft // 2 + 1) * sr / nfft


# ----------------------------------------------------------------- PLP (Hermansky 1990)
def hz2bark(f):
    return 6.0 * np.arcsinh(f / 600.0)


def bark2hz(z):
    return 600.0 * np.sinh(z / 6.0)


def plp_filterbank(nfft=NFFT, sr=SR):
    """Critical-band (Bark) integration, 1-Bark spacing, trapezoid-like Hermansky shape."""
    nfilts = int(math.ceil(hz2bark(sr / 2))) + 1
    step = hz2bark(sr / 2) / (nfilts - 1)
    binbarks = hz2bark(np.arange(nfft // 2 + 1) * sr / nfft)
    mids = np.arange(nfilts) * step
    wts = np.zeros((nfilts, nfft // 2 + 1))
    for i, mid in enumerate(mids):
        lof = binbarks - mid - 0.5
        hif = binbarks - mid + 0.5
        wts[i] = 10 ** np.minimum(0.0, np.minimum(hif, -2.5 * lof))
    return wts, mids


_PLP_FB, _PLP_MIDS = plp_filterbank()


def plp_cepstra(P, order=PLP_ORDER, n_cep=N_CEP):
    """Full PLP: Bark integration -> equal-loudness weighting -> cube-root intensity-loudness
    compression -> inverse DFT (autocorrelation) -> all-pole fit -> cepstra."""
    bands = P @ _PLP_FB.T                                    # critical-band integration
    f = bark2hz(_PLP_MIDS)
    fsq = f ** 2
    eql = ((fsq / (fsq + 1.6e5)) ** 2) * ((fsq + 1.44e6) / (fsq + 9.61e6))   # equal-loudness curve
    y = (bands * eql[None, :]) ** 0.33                       # intensity-loudness power law
    y[:, 0] = y[:, 1]                                        # replicate edge bands (as in RASTA-PLP)
    y[:, -1] = y[:, -2]
    sym = np.concatenate([y, y[:, -2:0:-1]], axis=1)         # symmetric auditory spectrum
    r = np.real(np.fft.ifft(sym, axis=1))[:, :order + 1]     # autocorrelation of auditory spectrum
    a, e = levinson_batch(r, order)
    return lpc_to_cep(a, n_cep), a, e


# ----------------------------------------------------------------- per-utterance API
def frame_features(x, preemph=True):
    """Frame-synchronous features for one 16 kHz signal.
    Speech-frame selection (VAD) is computed on the signal BEFORE pre-emphasis, so the same
    frames are compared with/without pre-emphasis. Returns dict of (T_speech, 13) arrays."""
    frames0 = frame_signal(x)
    mask = speech_mask(frames0)
    xs = preemphasis(x) if preemph else x
    P = power_spectrum(frame_signal(xs))
    a, e = lpc_from_power(P)
    plp, _, _ = plp_cepstra(P)
    out = {"MFCC": mfcc_from_power(P), "LPCC": lpc_to_cep(a), "PLP": plp}
    return {k: v[mask] for k, v in out.items()}


def utterance_vector(feats):
    """Per-stream utterance descriptor = [mean(13), std(13)] over speech frames."""
    return {k: np.concatenate([v.mean(0), v.std(0)]) for k, v in feats.items()}


In [ ]:
%%writefile run_experiment.py
"""
Speaker-independent speech-emotion recognition on RAVDESS (speech) comparing MFCC, LPC-cepstra and
PLP-cepstra, individually and fused, with/without pre-emphasis, clean and noisy test conditions.

Protocol
  * Features: frame-synchronous (16 kHz, 25/10 ms), per-utterance mean+std over speech frames (26-d/stream).
  * Fusion: early fusion = concatenation of utterance descriptors; z-scoring is fitted on TRAINING folds only.
  * Classifier: RBF-SVM, fixed hyper-parameters (C=10, gamma='scale'), no tuning (so no tuning leakage).
  * Split: leave-one-actor-out (24 folds) -> no actor ever appears in both train and test.
  * Noise: white Gaussian noise (SNR 20/10/0 dB) added to the audio, with two training protocols:
      - 'clean': train on clean speech only, test on clean + noisy (worst-case mismatch);
      - 'multi': train on clean + all noisy versions of the TRAINING actors, test on the held-out actor
                 at each condition (multi-condition training).
  * Statistics: paired Wilcoxon signed-rank over the per-actor accuracies, Holm-corrected within each family.

Usage:  python run_experiment.py --data_dir /path/to/RAVDESS --out results
"""
import argparse, glob, hashlib, itertools, os, time
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import wilcoxon
from sklearn.metrics import f1_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import features as F

EMOTIONS = {1: "neutral", 2: "calm", 3: "happy", 4: "sad", 5: "angry", 6: "fearful", 7: "disgust", 8: "surprised"}
CONFIGS = {
    "MFCC": ("MFCC",), "LPCC": ("LPCC",), "PLP": ("PLP",),
    "MFCC+LPCC": ("MFCC", "LPCC"), "MFCC+PLP": ("MFCC", "PLP"), "LPCC+PLP": ("LPCC", "PLP"),
    "MFCC+LPCC+PLP": ("MFCC", "LPCC", "PLP"),
}


def parse_name(path):
    p = os.path.basename(path)[:-4].split("-")
    if len(p) != 7 or p[1] != "01":          # keep speech (vocal channel 01); drop songs
        return None
    return dict(path=path, emotion=int(p[2]), actor=int(p[6]))


def seed_for(path, snr):
    return int(hashlib.md5(f"{os.path.basename(path)}|{snr}".encode()).hexdigest()[:8], 16)


def extract_clip(path, snrs):
    """All (pre-emphasis, condition) feature vectors for one clip."""
    x = F.load_audio(path)
    out = {}
    for cond in ["clean"] + list(snrs):
        xc = x if cond == "clean" else F.add_noise(x, cond, np.random.default_rng(seed_for(path, cond)))
        for pe in (False, True):
            out[(pe, cond)] = F.utterance_vector(F.frame_features(xc, preemph=pe))
    return out


def holm(p):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--out", default="results")
    ap.add_argument("--snrs", type=float, nargs="*", default=[20, 10, 0])
    ap.add_argument("--jobs", type=int, default=-1)
    ap.add_argument("--protocols", nargs="*", default=["clean", "multi"], choices=["clean", "multi"])
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    meta = [m for m in (parse_name(p) for p in sorted(glob.glob(os.path.join(a.data_dir, "**", "*.wav"), recursive=True))) if m]
    assert meta, "No RAVDESS speech files (03-01-*-*-*-*-*.wav) found."
    y = np.array([m["emotion"] for m in meta]); g = np.array([m["actor"] for m in meta])
    print(f"{len(meta)} clips, {len(set(g))} actors, {len(set(y))} emotions")

    t0 = time.time()
    feats = Parallel(n_jobs=a.jobs, verbose=5)(delayed(extract_clip)(m["path"], a.snrs) for m in meta)
    print(f"feature extraction done in {time.time() - t0:.0f}s")

    conds = ["clean"] + list(a.snrs)
    def X_for(pe, cond, cfg):
        return np.array([np.concatenate([f[(pe, cond)][s] for s in CONFIGS[cfg]]) for f in feats])

    rows = []
    for pe, cfg in itertools.product((False, True), CONFIGS):
        Xte = {c: X_for(pe, c, cfg) for c in conds}
        for tr, te in LeaveOneGroupOut().split(Xte["clean"], y, g):
            for tm in a.protocols:
                if tm == "clean":
                    Xfit, yfit = Xte["clean"][tr], y[tr]
                else:
                    Xfit = np.vstack([Xte[c][tr] for c in conds]); yfit = np.tile(y[tr], len(conds))
                clf = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale", random_state=0))
                clf.fit(Xfit, yfit)
                for c in conds:
                    pred = clf.predict(Xte[c][te])
                    rows.append(dict(train=tm, preemph=pe, config=cfg, cond=str(c), actor=int(g[te][0]),
                                     acc=float(np.mean(pred == y[te])),
                                     f1=float(f1_score(y[te], pred, average="macro", labels=sorted(EMOTIONS), zero_division=0))))
        print("done", "preemph" if pe else "raw", cfg, flush=True)
    df = pd.DataFrame(rows); df.to_csv(os.path.join(a.out, "per_actor_results.csv"), index=False)

    summ = (df.groupby(["train", "preemph", "config", "cond"])[["acc", "f1"]].agg(["mean", "std"]).round(4))
    summ.columns = ["acc_mean", "acc_std", "f1_mean", "f1_std"]; summ = summ.reset_index()
    summ.to_csv(os.path.join(a.out, "summary.csv"), index=False)

    # ---- significance (paired over actors; Holm within each family)
    def paired(d1, d2):
        m = d1.merge(d2, on="actor", suffixes=("_1", "_2"))
        diff = m["acc_1"] - m["acc_2"]
        p = 1.0 if np.allclose(diff, 0) else wilcoxon(m["acc_1"], m["acc_2"]).pvalue
        return diff.mean(), p
    srows = []
    for tm in a.protocols:
        dt = df[df.train == tm]
        for pe in (False, True):                                    # family A: fusion vs MFCC baseline
            for cond in map(str, conds):
                fam = []
                for cfg in CONFIGS:
                    if cfg == "MFCC": continue
                    d1 = dt[(dt.preemph == pe) & (dt.config == cfg) & (dt.cond == cond)]
                    d2 = dt[(dt.preemph == pe) & (dt.config == "MFCC") & (dt.cond == cond)]
                    md, p = paired(d1, d2); fam.append(dict(train=tm, family="fusion_vs_MFCC", preemph=pe, cond=cond,
                                                            comparison=f"{cfg} - MFCC", mean_diff_acc=md, p=p))
                ph = holm([r["p"] for r in fam])
                for r, pa in zip(fam, ph): r["p_holm"] = pa
                srows += fam
        for cond in map(str, conds):                                # family B: pre-emphasis effect
            fam = []
            for cfg in CONFIGS:
                d1 = dt[(dt.preemph) & (dt.config == cfg) & (dt.cond == cond)]
                d0 = dt[(~dt.preemph) & (dt.config == cfg) & (dt.cond == cond)]
                md, p = paired(d1, d0); fam.append(dict(train=tm, family="preemph_effect", preemph="on-off", cond=cond,
                                                        comparison=cfg, mean_diff_acc=md, p=p))
            ph = holm([r["p"] for r in fam])
            for r, pa in zip(fam, ph): r["p_holm"] = pa
            srows += fam
    pd.DataFrame(srows).round(4).to_csv(os.path.join(a.out, "significance.csv"), index=False)

    # ---- console tables
    for tm in a.protocols:
        for pe, lab in ((True, "ON"), (False, "OFF")):
            piv = summ[(summ.train == tm) & (summ.preemph == pe)].pivot(index="config", columns="cond", values="acc_mean")[[str(c) for c in conds]]
            print(f"\nAccuracy | train={tm} | pre-emphasis {lab} (mean over actors):\n", piv.round(3))
    print(f"\nChance level = {1/len(set(y)):.3f}.  Results written to {a.out}/")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile make_figures.py
"""
Publication figures/tables for the revised manuscript (white background, 300 dpi, time in seconds,
shared colour limits for compared panels, labelled colour bars).

Usage:  python make_figures.py --wav /path/to/one_RAVDESS_clip.wav --out figures [--results results]
"""
import argparse, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import features as F

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
                     "font.size": 10, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 9,
                     "savefig.dpi": 300, "figure.dpi": 100})
HOPS = F.HOP / F.SR


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--wav", required=True); ap.add_argument("--out", default="figures")
    ap.add_argument("--results", default=None)
    a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)

    x = F.load_audio(a.wav); xe = F.preemphasis(x)
    t = np.arange(len(x)) / F.SR
    mask = F.speech_mask(F.frame_signal(x))

    # ---------------- Fig 1: waveforms (shared amplitude axis, seconds)
    fig, ax = plt.subplots(2, 1, figsize=(6.5, 4.2), sharex=True, sharey=True)
    ax[0].plot(t, x, lw=0.4, color="k"); ax[0].set_title("(a) Original speech")
    ax[1].plot(t, xe, lw=0.4, color="k"); ax[1].set_title(r"(b) Pre-emphasised ($\alpha$ = 0.97)")
    for a_ in ax: a_.set_ylabel("Amplitude (a.u.)")
    ax[1].set_xlabel("Time (s)"); fig.tight_layout(); fig.savefig(f"{a.out}/fig1_waveforms.png"); plt.close(fig)

    # ---------------- features (speech frames only for statistics)
    def all_frames(sig):
        P = F.power_spectrum(F.frame_signal(sig)); return P
    P0, P1 = all_frames(x), all_frames(xe)
    M0, M1 = F.mfcc_from_power(P0), F.mfcc_from_power(P1)

    # ---------------- Fig 2: MFCC maps, SHARED colour limits
    vmin = min(M0.min(), M1.min()); vmax = max(M0.max(), M1.max())
    fig, ax = plt.subplots(1, 2, figsize=(7.0, 3.2), sharey=True)
    for a_, M, ttl in zip(ax, (M0, M1), ("(a) Original", "(b) Pre-emphasised")):
        im = a_.imshow(M.T, origin="lower", aspect="auto", vmin=vmin, vmax=vmax, cmap="viridis",
                       extent=[0, M.shape[0] * HOPS, 0.5, F.N_CEP + 0.5])
        a_.set_title(ttl); a_.set_xlabel("Time (s)")
    ax[0].set_ylabel("Cepstral coefficient index")
    cb = fig.colorbar(im, ax=ax, label="MFCC value (dimensionless)"); fig.savefig(f"{a.out}/fig2_mfcc_maps.png", bbox_inches="tight"); plt.close(fig)

    # ---------------- Fig 3: spectral envelopes at the highest-energy speech frame
    e = 10 * np.log10(np.sum(F.frame_signal(x) ** 2, axis=1) + 1e-12); e[~mask] = -np.inf
    k = int(np.argmax(e))
    fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True)
    for a_, P, ttl in zip(ax, (P0, P1), ("(a) Original", "(b) Pre-emphasised")):
        dft = 10 * np.log10(P[k] + 1e-20)
        al, el = F.lpc_from_power(P[k:k + 1]); lpc_env, fax = F.lpc_envelope_db(al, el)
        _, ap_, ep = F.plp_cepstra(P[k:k + 1]); plp_env, _ = F.lpc_envelope_db(ap_, ep)
        lpc_env = lpc_env[0] - lpc_env[0].mean() + dft.mean()
        plp_env = plp_env[0] - plp_env[0].mean() + dft.mean()       # levels aligned to the DFT mean for shape comparison
        a_.plot(fax / 1000, dft, color="0.75", lw=0.8, label="DFT (frame)")
        a_.plot(fax / 1000, lpc_env, color="C0", lw=1.5, label=f"LPC, order {F.LPC_ORDER}")
        a_.plot(fax / 1000, plp_env, color="C3", lw=1.5, ls="--", label=f"PLP, order {F.PLP_ORDER}")
        a_.set_title(ttl); a_.set_xlabel("Frequency (kHz)"); a_.set_xlim(0, 8)
    ax[0].set_ylabel("Level (dB, envelopes level-aligned)"); ax[0].legend(loc="upper right")
    fig.tight_layout(); fig.savefig(f"{a.out}/fig3_envelopes.png"); plt.close(fig)

    # ---------------- Table 1 (replacement): per-coefficient mean/std across speech frames
    rows = []
    for i in range(F.N_CEP):
        rows.append({"coef": f"C{i+1}",
                     "raw_mean": M0[mask, i].mean(), "raw_std": M0[mask, i].std(),
                     "preemph_mean": M1[mask, i].mean(), "preemph_std": M1[mask, i].std()})
    pd.DataFrame(rows).round(3).to_csv(f"{a.out}/table1_mfcc_stats.csv", index=False)

    # ---------------- Pre-emphasis quantified: spectral slope & centroid
    f = np.arange(F.NFFT // 2 + 1) * F.SR / F.NFFT; band = (f >= 100) & (f <= 8000)
    def slope_centroid(P):
        sl = [np.polyfit(np.log2(f[band]), 10 * np.log10(p[band] + 1e-20), 1)[0] for p in P[mask]]
        ce = (P[mask] * f).sum(1) / (P[mask].sum(1) + 1e-20)
        return np.array(sl), ce
    s0, c0 = slope_centroid(P0); s1, c1 = slope_centroid(P1)
    pd.DataFrame({"measure": ["spectral slope (dB/octave, 100-8000 Hz)", "spectral centroid (Hz)"],
                  "raw_mean": [s0.mean(), c0.mean()], "raw_std": [s0.std(), c0.std()],
                  "preemph_mean": [s1.mean(), c1.mean()], "preemph_std": [s1.std(), c1.std()]}
                 ).round(2).to_csv(f"{a.out}/table_spectral_tilt.csv", index=False)

    # ---------------- Fig 4: accuracy vs condition (from run_experiment.py output)
    if a.results:
        S = pd.read_csv(os.path.join(a.results, "summary.csv")); S["cond"] = S["cond"].astype(str)
        protos = list(S["train"].unique()); conds = ["clean"] + [c for c in S["cond"].unique() if c != "clean"]
        conds = ["clean"] + sorted([c for c in conds if c != "clean"], key=lambda v: -float(v))
        fig, ax = plt.subplots(1, len(protos), figsize=(3.6 * len(protos) + 0.8, 3.4), sharey=True, squeeze=False)
        for a_, tm in zip(ax[0], protos):
            for cfg in S.config.unique():
                d = S[(S.train == tm) & (S.preemph) & (S.config == cfg)].set_index("cond").loc[conds]
                a_.errorbar(range(len(conds)), d.acc_mean, yerr=d.acc_std / np.sqrt(24), marker="o", ms=3, lw=1, capsize=2, label=cfg)
            a_.set_xticks(range(len(conds))); a_.set_xticklabels([c if c == "clean" else f"{c} dB" for c in conds])
            a_.set_title(f"Training: {'clean only' if tm == 'clean' else 'multi-condition'}"); a_.set_xlabel("Test condition (SNR)")
            a_.axhline(1 / 8, color="k", ls=":", lw=0.8)
        ax[0][0].set_ylabel("Accuracy (mean over actors; bars = s.e.m.)"); ax[0][-1].legend(fontsize=7, loc="upper right")
        fig.tight_layout(); fig.savefig(f"{a.out}/fig4_accuracy_vs_snr.png"); plt.close(fig)
    print("saved to", a.out)


if __name__ == "__main__":
    main()


## Download RAVDESS speech (Zenodo record 1188976)

In [ ]:
!wget -q --show-progress -O ravdess.zip 'https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1'
!mkdir -p ravdess && unzip -q -o ravdess.zip -d ravdess
!find ravdess -name '*.wav' | wc -l   # expect 1440

## Run the experiment (slow)

In [ ]:
!python run_experiment.py --data_dir ravdess --out results --jobs -1

## Figures and tables

In [ ]:
import glob
wav = sorted(glob.glob('ravdess/**/03-01-05-01-01-01-03.wav', recursive=True))[0]
!python make_figures.py --wav {wav} --out figures --results results

## Zip and download

In [ ]:
!zip -qr results_and_figures.zip results figures
from google.colab import files
files.download('results_and_figures.zip')